### 序列分类任务

#### 加载预训练模型和tokenizer

In [1]:
import os
# 设置环境变量，此处使用 HuggingFace 镜像网站
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
# # 下载模型
# os.system('huggingface-cli download --resume-download Qwen/Qwen2.5-1.5B --local-dir your_local_dir')

from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained('facebook/esm2_t33_650M_UR50D')
model = AutoModelForSequenceClassification.from_pretrained('facebook/esm2_t33_650M_UR50D', num_labels=2)

print('Model and Tokenizer loaded successfully!')

/root/miniconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 566/566 [00:00<00:00, 3408.07it/s]
EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different ta

Model and Tokenizer loaded successfully!


#### 准备数据，序列和亚细胞定位

In [2]:
import requests

query_url ="https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Csequence%2Ccc_subcellular_location&format=tsv&query=%28%28organism_id%3A9606%29%20AND%20%28reviewed%3Atrue%29%20AND%20%28length%3A%5B80%20TO%20500%5D%29%29"
uniprot_request = requests.get(query_url)

In [3]:
from io import BytesIO
import pandas as pd
#
# 使用一个BytesIO对象，Pandas会将其当作文件处理。
# 如果将数据下载为文件，则可以跳过这一步，直接将文件路径传递给read_csv
bio = BytesIO(uniprot_request.content)
# 读取数据，分隔符为制表符
df = pd.read_csv(bio, compression='gzip', sep='\t')
df

,Entry,Sequence,Subcellular location [CC]
0,A0A096LP01,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,SUBCELLULAR LOCATION: Mitochondrion outer memb...
1,A0A0K2S4Q6,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,SUBCELLULAR LOCATION: [Isoform 1]: Membrane {E...
2,A0A2R8Y7D0,MEGLRRGLSRWKRYHIKVHLADEALLLPLTVRPRDTLSDLRAQLVG...,SUBCELLULAR LOCATION: Nucleus {ECO:0000269|Pub...
3,A0A8I5KQE6,MSGALDVLQMKEEDVLKFLAAGTHLGGTNLDFQMEHYIYKRKSDGI...,SUBCELLULAR LOCATION: Cell membrane {ECO:00002...
4,A0AVI4,MDSPEVTFTLAYLVFAVCFVFTPNEFHAAGLTVQNLLSGWLGSEDA...,SUBCELLULAR LOCATION: Endoplasmic reticulum me...
...,...,...,...
11977,Q9HAA7,MLFGIRILVNTPSPLVTGLHHYNPSIHRDQGECANQWRKGPGSAHL...,NaN
11978,Q9NZ38,MAFPGQSDTKMQWPEVPALPLLSSLCMAMVRKSSALGKEVGRRSEG...,NaN
11979,Q9UFV3,MAETYRRSRQHEQLPGQRHMDLLTGYSKLIQSRLKLLLHLGSQPPV...,NaN
11980,Q9Y4M8,MATFHRAHATSSVKPRARRHQEPNSGDWPGSYRAGTRCSAIGFRLL...,NaN


#### 处理数据

In [4]:
df = df.dropna()
df.shape

cytosolic = df['Subcellular location [CC]'].str.contains('Cytosol') | df['Subcellular location [CC]'].str.contains('Cytoplasm')
membrane = df['Subcellular location [CC]'].str.contains('Membrane') | df['Subcellular location [CC]'].str.contains('Membraneous')

df_cytosolic = df[cytosolic & ~membrane] # 细胞质蛋白
df_membrane = df[membrane & ~cytosolic] # 膜蛋白
df_cytosolic.shape, df_membrane.shape

((3081, 3), (813, 3))

In [5]:
cytosolic_sequences = df_cytosolic['Sequence'].tolist()
membrane_sequences = df_membrane['Sequence'].tolist()

cytosolic_labels = [0] * len(cytosolic_sequences) # 细胞质蛋白标签为0
membrane_labels = [1] * len(membrane_sequences) # 膜蛋白标签为1

sequences = cytosolic_sequences + membrane_sequences
labels = cytosolic_labels + membrane_labels

# 检查
print(f"Total sequences: {len(sequences)}")
print(f"Total labels: {len(labels)}")

Total sequences: 3894
Total labels: 3894


#### 划分数据

In [6]:
from sklearn.model_selection import train_test_split

train_seqs, val_seqs, train_labels, val_labels = train_test_split(sequences, labels, test_size=0.2, random_state=42)
print(f"Training sequences: {len(train_seqs)}, Training labels: {len(train_labels)}")
print(f"Validation sequences: {len(val_seqs)}, Validation labels: {len(val_labels)}")
print(f'train_seqs sample: {train_seqs[0]}, train_labels sample: {train_labels[0]}')

Training sequences: 3115, Training labels: 3115
Validation sequences: 779, Validation labels: 779
train_seqs sample: MAGYATTPSPMQTLQEEAVCAICLDYFKDPVSISCGHNFCRGCVTQLWSKEDEEDQNEEEDEWEEEEDEEAVGAMDGWDGSIREVLYRGNADEELFQDQDDDELWLGDSGITNWDNVDYMWDEEEEEEEEDQDYYLGGLRPDLRIDVYREEEILEAYDEDEDEELYPDIHPPPSLPLPGQFTCPQCRKSFTRRSFRPNLQLANMVQIIRQMCPTPYRGNRSNDQGMCFKHQEALKLFCEVDKEAICVVCRESRSHKQHSVLPLEEVVQEYQEIKLETTLVGILQIEQESIHSKAYNQ, train_labels sample: 0


#### Tokenizing数据

In [11]:
print(tokenizer(train_seqs[0]))

train_tokenized = tokenizer(train_seqs, truncation=True, max_length=512)
val_tokenized = tokenizer(val_seqs, truncation=True, max_length=512)

{'input_ids': [0, 20, 5, 6, 19, 5, 11, 11, 14, 8, 14, 20, 16, 11, 4, 16, 9, 9, 5, 7, 23, 5, 12, 23, 4, 13, 19, 18, 15, 13, 14, 7, 8, 12, 8, 23, 6, 21, 17, 18, 23, 10, 6, 23, 7, 11, 16, 4, 22, 8, 15, 9, 13, 9, 9, 13, 16, 17, 9, 9, 9, 13, 9, 22, 9, 9, 9, 9, 13, 9, 9, 5, 7, 6, 5, 20, 13, 6, 22, 13, 6, 8, 12, 10, 9, 7, 4, 19, 10, 6, 17, 5, 13, 9, 9, 4, 18, 16, 13, 16, 13, 13, 13, 9, 4, 22, 4, 6, 13, 8, 6, 12, 11, 17, 22, 13, 17, 7, 13, 19, 20, 22, 13, 9, 9, 9, 9, 9, 9, 9, 9, 13, 16, 13, 19, 19, 4, 6, 6, 4, 10, 14, 13, 4, 10, 12, 13, 7, 19, 10, 9, 9, 9, 12, 4, 9, 5, 19, 13, 9, 13, 9, 13, 9, 9, 4, 19, 14, 13, 12, 21, 14, 14, 14, 8, 4, 14, 4, 14, 6, 16, 18, 11, 23, 14, 16, 23, 10, 15, 8, 18, 11, 10, 10, 8, 18, 10, 14, 17, 4, 16, 4, 5, 17, 20, 7, 16, 12, 12, 10, 16, 20, 23, 14, 11, 14, 19, 10, 6, 17, 10, 8, 17, 13, 16, 6, 20, 23, 18, 15, 21, 16, 9, 5, 4, 15, 4, 18, 23, 9, 7, 13, 15, 9, 5, 12, 23, 7, 7, 23, 10, 9, 8, 10, 8, 21, 15, 16, 21, 8, 7, 4, 14, 4, 9, 9, 7, 7, 16, 9, 19, 16, 9, 12, 15, 4

#### 加载数据集

In [8]:

from datasets import Dataset

train_dataset = Dataset.from_dict(train_tokenized)
val_dataset = Dataset.from_dict(val_tokenized)

print(train_dataset[0])
print(val_dataset[0])

train_dataset = train_dataset.add_column('labels', train_labels)
val_dataset = val_dataset.add_column('labels', val_labels)

print(train_dataset[0])
print(val_dataset[0])

{'input_ids': [0, 20, 5, 6, 19, 5, 11, 11, 14, 8, 14, 20, 16, 11, 4, 16, 9, 9, 5, 7, 23, 5, 12, 23, 4, 13, 19, 18, 15, 13, 14, 7, 8, 12, 8, 23, 6, 21, 17, 18, 23, 10, 6, 23, 7, 11, 16, 4, 22, 8, 15, 9, 13, 9, 9, 13, 16, 17, 9, 9, 9, 13, 9, 22, 9, 9, 9, 9, 13, 9, 9, 5, 7, 6, 5, 20, 13, 6, 22, 13, 6, 8, 12, 10, 9, 7, 4, 19, 10, 6, 17, 5, 13, 9, 9, 4, 18, 16, 13, 16, 13, 13, 13, 9, 4, 22, 4, 6, 13, 8, 6, 12, 11, 17, 22, 13, 17, 7, 13, 19, 20, 22, 13, 9, 9, 9, 9, 9, 9, 9, 9, 13, 16, 13, 19, 19, 4, 6, 6, 4, 10, 14, 13, 4, 10, 12, 13, 7, 19, 10, 9, 9, 9, 12, 4, 9, 5, 19, 13, 9, 13, 9, 13, 9, 9, 4, 19, 14, 13, 12, 21, 14, 14, 14, 8, 4, 14, 4, 14, 6, 16, 18, 11, 23, 14, 16, 23, 10, 15, 8, 18, 11, 10, 10, 8, 18, 10, 14, 17, 4, 16, 4, 5, 17, 20, 7, 16, 12, 12, 10, 16, 20, 23, 14, 11, 14, 19, 10, 6, 17, 10, 8, 17, 13, 16, 6, 20, 23, 18, 15, 21, 16, 9, 5, 4, 15, 4, 18, 23, 9, 7, 13, 15, 9, 5, 12, 23, 7, 7, 23, 10, 9, 8, 10, 8, 21, 15, 16, 21, 8, 7, 4, 14, 4, 9, 9, 7, 7, 16, 9, 19, 16, 9, 12, 15, 4

#### 训练模型

In [16]:
import torch
import accelerate
import transformers
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

args = TrainingArguments(
    output_dir= './sft_output/esm2_t33_650M_UR50D-finetuned-subcellular-location',
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True, # 训练结束后加载最佳模型
    metric_for_best_model='accuracy',
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
def compute_metrics(pred):
    labels = pred.label_ids # 获取真实标签
    preds = pred.predictions.argmax(-1) # 获取预测结果，argmax(-1)表示取最后一个维度的最大值索引，即预测的类别  
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary') # 计算精确率、召回率和F1分数，average='binary'表示二分类任务    
    acc = accuracy_score(labels, preds) # 计算准确率
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator)


[RANK 0] Detected kernel version 5.4.250, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [17]:
trainer.train()

/root/miniconda3/envs/llm/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.203998,0.924262,0.769697,0.858108,0.811502
2,No log,0.188397,0.933248,0.800000,0.864865,0.831169
3,No log,0.189417,0.931964,0.798742,0.858108,0.827362


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.22s/it]
/root/miniconda3/envs/llm/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.35s/it]
/root/miniconda3/envs/llm/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.34s/it]
There were missing keys in the checkpoint model loaded: ['esm.encoder.layer.0.attention.LayerNorm.weight', 'esm.encoder.layer.0.attention.LayerNorm.bias', 'esm.encoder.layer.0.LayerNorm.weight', 'esm.encoder.layer.0.LayerNorm.bias', 'esm.encoder.layer.1.attention.LayerNorm.weight', 'esm.encoder.layer.1.attention.La

TrainOutput(global_step=147, training_loss=0.22695294853781356, metrics={'train_runtime': 151.69, 'train_samples_per_second': 61.606, 'train_steps_per_second': 0.969, 'total_flos': 1.8019026262411784e+16, 'train_loss': 0.22695294853781356, 'epoch': 3.0})

### Token分类任务

#### 准备数据

In [19]:
import requests

query_url ="https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Csequence%2Cft_strand%2Cft_helix&format=tsv&query=%28%28organism_id%3A9606%29%20AND%20%28reviewed%3Atrue%29%20AND%20%28length%3A%5B80%20TO%20500%5D%29%29"
uniprot_request = requests.get(query_url)

In [20]:
from io import BytesIO
import pandas as pd
# 使用一个BytesIO对象，Pandas会将其当作文件处理。
# 如果将数据下载为文件，则可以跳过这一步，直接将文件路径传递给read_csv
bio = BytesIO(uniprot_request.content)
# 读取数据，分隔符为制表符
df = pd.read_csv(bio, compression='gzip', sep='\t')
df


,Entry,Sequence,Beta strand,Helix
0,A0A096LP01,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,NaN,NaN
1,A0A0K2S4Q6,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,NaN,NaN
2,A0A2R8Y7D0,MEGLRRGLSRWKRYHIKVHLADEALLLPLTVRPRDTLSDLRAQLVG...,"STRAND 14..20; /evidence=""ECO:0007829|PDB:7MRJ...","HELIX 4..7; /evidence=""ECO:0007829|PDB:7MRJ""; ..."
3,A0A8I5KQE6,MSGALDVLQMKEEDVLKFLAAGTHLGGTNLDFQMEHYIYKRKSDGI...,NaN,NaN
4,A0AVI4,MDSPEVTFTLAYLVFAVCFVFTPNEFHAAGLTVQNLLSGWLGSEDA...,NaN,NaN
...,...,...,...,...
11977,Q9HAA7,MLFGIRILVNTPSPLVTGLHHYNPSIHRDQGECANQWRKGPGSAHL...,NaN,NaN
11978,Q9NZ38,MAFPGQSDTKMQWPEVPALPLLSSLCMAMVRKSSALGKEVGRRSEG...,NaN,NaN
11979,Q9UFV3,MAETYRRSRQHEQLPGQRHMDLLTGYSKLIQSRLKLLLHLGSQPPV...,NaN,NaN
11980,Q9Y4M8,MATFHRAHATSSVKPRARRHQEPNSGDWPGSYRAGTRCSAIGFRLL...,NaN,NaN


#### 处理数据

In [21]:
no_stru_rows = df['Beta strand'].isna() & df['Helix'].isna()
df = df[~no_stru_rows]
df

,Entry,Sequence,Beta strand,Helix
2,A0A2R8Y7D0,MEGLRRGLSRWKRYHIKVHLADEALLLPLTVRPRDTLSDLRAQLVG...,"STRAND 14..20; /evidence=""ECO:0007829|PDB:7MRJ...","HELIX 4..7; /evidence=""ECO:0007829|PDB:7MRJ""; ..."
5,A0JLT2,MENFTALFGAQADPPPPPTALGFGPGKPPPPPPPPAGGGPGTAPPP...,"STRAND 79..81; /evidence=""ECO:0007829|PDB:7EMF""","HELIX 83..86; /evidence=""ECO:0007829|PDB:7EMF""..."
17,A1L3X0,MAFSDLTSRTVHLYDNWIKDADPRVEDWLLMSSPLPQTILLGFYVY...,"STRAND 97..99; /evidence=""ECO:0007829|PDB:6Y7F""","HELIX 17..20; /evidence=""ECO:0007829|PDB:6Y7F""..."
18,A1XBS5,MMRRTLENRNAQTKQLQTAVSNVEKHFGELCQIFAAYVRKTARLRD...,NaN,"HELIX 2..6; /evidence=""ECO:0007829|PDB:8CEG""; ..."
19,A1Z1Q3,MYPSNKKKKVWREEKERLLKMTLEERRKEYLRDYIPLNSILSWKEE...,"STRAND 71..77; /evidence=""ECO:0007829|PDB:4IQY...","HELIX 11..19; /evidence=""ECO:0007829|PDB:4IQY""..."
...,...,...,...,...
11604,Q96I45,MVNLGLSRVDDAVAAKHPGLGEYAACQSHAFMKGVFTFVTGTGMAF...,"STRAND 3..5; /evidence=""ECO:0007829|PDB:2LOR"";...","HELIX 6..16; /evidence=""ECO:0007829|PDB:2LOR"";..."
11658,Q9H0W7,MPTNCAAAGCATTYNKHINISFHRFPLDPKRRKEWVRLVRRKNFVP...,"STRAND 7..9; /evidence=""ECO:0007829|PDB:2D8R"";...","HELIX 29..38; /evidence=""ECO:0007829|PDB:2D8R"""
11695,Q9P1F3,MNVDHEVNLLVEEIHRLGSKNADGKLSVKFGVLFRDDKCANLFEAL...,"STRAND 24..29; /evidence=""ECO:0007829|PDB:2L2O...","HELIX 3..17; /evidence=""ECO:0007829|PDB:2L2O"";..."
11697,Q9P298,MSANRRWWVPPDDEDCVSEKLLRKTRESPLVPIGLGGCLVVAAYRI...,"STRAND 11..14; /evidence=""ECO:0007829|PDB:2LON...","HELIX 18..24; /evidence=""ECO:0007829|PDB:2LON""..."


In [ ]:
df.iloc[0]['Helix']

'HELIX 4..7; /evidence="ECO:0007829|PDB:7MRJ"; HELIX 37..46; /evidence="ECO:0007829|PDB:7MRJ"'

In [23]:
import re

strand_re = r'STRAND\s(\d+)\.\.(\d+)\;' # 匹配STRAND后面跟着的两个数字，格式为"STRAND 10..20;"，其中10和20是起始和结束位置
helix_re = r'HELIX\s(\d+)\.\.(\d+)\;' # 匹配HELIX后面跟着的两个数字，格式为"HELIX 10..20;"，其中10和20是起始和结束位置

re.findall(strand_re, df.iloc[0]['Beta strand'])
re.findall(helix_re, df.iloc[0]['Helix'])

[('4', '7'), ('37', '46')]

In [24]:
import numpy as np

def build_labels(seq, strands, helices):
    labels = np.zeros(len(seq), dtype=int) # 初始化标签为0，表示无结构
    if isinstance(helices, float): # 判断helices是否为字符串，避免空值导致错误
        found_helices = []
    else:
        found_helices = re.findall(helix_re, helices)
        for helix in found_helices:
            helix_start, helix_end = map(int, helix)
            assert helix_end <= len(seq)
            labels[helix_start-1:helix_end] = 1 # HELIX标签为1

    if isinstance(strands, float): # 判断strands是否为字符串，避免空值导致错误
        found_strands = []
    else:
        found_strands = re.findall(strand_re, strands)
        for strand in found_strands:
            strand_start, strand_end = map(int, strand)
            assert strand_end <= len(seq)
            labels[strand_start-1:strand_end] = 2 # STRAND标签为2
    return labels

In [25]:
seq = []
labels = []

for raw_idx, row in df.iterrows():
    raw_seq = row['Sequence']
    raw_strands = row['Beta strand']
    raw_helices = row['Helix']
    seq.append(raw_seq)
    labels.append(build_labels(raw_seq, raw_strands, raw_helices))

In [29]:
labels[0]

array([0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 0, 0,
       0, 0, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 0, 0, 0])

#### 构建数据集

In [30]:
from sklearn.model_selection import train_test_split

train_seqs, val_seqs, train_labels, val_labels = train_test_split(seq, labels, test_size=0.2, random_state=42)
print(f"Training sequences: {len(train_seqs)}, Training labels: {len(train_labels)}")
print(f"Validation sequences: {len(val_seqs)}, Validation labels: {len(val_labels)}")
print(f'train_seqs sample: {train_seqs[0]}, train_labels sample: {train_labels[0]}')

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('facebook/esm2_t33_650M_UR50D')
train_tokenized = tokenizer(train_seqs, truncation=True, max_length=512)
val_tokenized = tokenizer(val_seqs, truncation=True, max_length=512)

from datasets import Dataset
train_dataset = Dataset.from_dict(train_tokenized)
val_dataset = Dataset.from_dict(val_tokenized)
train_dataset = train_dataset.add_column('labels', train_labels)
val_dataset = val_dataset.add_column('labels', val_labels)
print(train_dataset[0])
print(val_dataset[0])


Training sequences: 3514, Training labels: 3514
Validation sequences: 879, Validation labels: 879
train_seqs sample: MDPLRAQQLAAELEVEMMADMYNRMTSACHRKCVPPHYKEAELSKGESVCLDRCVSKYLDIHERMGKKLTELSMQDEELMKRVQQSSGPA, train_labels sample: [0 0 0 0 1 1 1 1 1 1 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0
 0 0 2 2 2 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 0 0 0 0 0 1 1 1 1 1 1 1 0 0 0 0]
{'input_ids': [0, 20, 13, 14, 4, 10, 5, 16, 16, 4, 5, 5, 9, 4, 9, 7, 9, 20, 20, 5, 13, 20, 19, 17, 10, 20, 11, 8, 5, 23, 21, 10, 15, 23, 7, 14, 14, 21, 19, 15, 9, 5, 9, 4, 8, 15, 6, 9, 8, 7, 23, 4, 13, 10, 23, 7, 8, 15, 19, 4, 13, 12, 21, 9, 10, 20, 6, 15, 15, 4, 11, 9, 4, 8, 20, 16, 13, 9, 9, 4, 20, 15, 10, 7, 16, 16, 8, 8, 6, 14, 5, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

#### 加载模型

In [31]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification

num_labels = 3 # 0: 无结构, 1: HELIX, 2: STRAND

model = AutoModelForTokenClassification.from_pretrained('facebook/esm2_t33_650M_UR50D', num_labels=num_labels)
print('Model loaded successfully!')

data_collator = DataCollatorForTokenClassification(tokenizer)

# 特性	DataCollatorWithPadding	DataCollatorForTokenClassification
# 主要用途	通用任务（分类、生成）	序列标注任务（NER等）
# 标签处理	不特别处理标签	正确处理标签，避免标签填充
# 填充策略	动态填充输入序列	同时处理输入和标签的填充对齐
# 标签填充值	通常不涉及	可设置（如-100表示忽略）
# 适用任务	文本分类、文本生成	命名实体识别、词性标注等

Loading weights: 100%|██████████| 566/566 [00:00<00:00, 7844.64it/s]
EsmForTokenClassification LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!


In [33]:
args = TrainingArguments(
    './sft_output/esm2_t33_650M_UR50D-finetuned-secondary-structure',
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.001,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)


def compute_metrics_combined(pred):
    """适用于大多数情况的评估函数"""
    predictions, labels = pred
    
    # 处理不同维度
    if predictions.ndim == 3:  # 序列标注任务
        predictions = np.argmax(predictions, axis=2)
        predictions = predictions.reshape(-1)
        labels = labels.reshape(-1)
        
        # 过滤填充标签（如果存在）
        mask = labels != -100
        if mask.any():
            predictions = predictions[mask]
            labels = labels[mask]
    else:  # 分类任务
        predictions = np.argmax(predictions, axis=1)
    
    # 计算多个指标
    acc = (predictions == labels).mean()
    
    # 计算其他指标（如果标签类别合适）
    unique_labels = np.unique(labels)
    if len(unique_labels) > 1 and len(unique_labels) <= 20:  # 避免过多类别
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='macro', zero_division=0
        )
        return {
            'accuracy': acc,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        }
    else:
        return {'accuracy': acc}

In [34]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_combined,
    data_collator=data_collator,
)

trainer.train()

[RANK 0] Detected kernel version 5.4.250, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/root/miniconda3/envs/llm/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.418484,0.833246,0.831844,0.790787,0.807827
2,No log,0.405972,0.838821,0.834320,0.803415,0.816895
3,No log,0.421467,0.840955,0.836722,0.806189,0.819590


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.32s/it]
/root/miniconda3/envs/llm/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.44s/it]
/root/miniconda3/envs/llm/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.57s/it]
There were missing keys in the checkpoint model loaded: ['esm.encoder.layer.0.attention.LayerNorm.weight', 'esm.encoder.layer.0.attention.LayerNorm.bias', 'esm.encoder.layer.0.LayerNorm.weight', 'esm.encoder.layer.0.LayerNorm.bias', 'esm.encoder.layer.1.attention.LayerNorm.weight', 'esm.encoder.layer.1.attention.La

TrainOutput(global_step=165, training_loss=0.37619686704693417, metrics={'train_runtime': 167.687, 'train_samples_per_second': 62.867, 'train_steps_per_second': 0.984, 'total_flos': 2.0277466186494816e+16, 'train_loss': 0.37619686704693417, 'epoch': 3.0})